In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import lingam
from lingam.utils import make_dot
from cdt.metrics import SHD
from causal_comparator.utils import SHD_vectorized

In [ ]:
import pandas as pd
import numpy as np
from causal_comparator.data_generation import EdgePerturbationSimulator
from causal_comparator.discovery import CausalComparator
from causal_comparator.metrics import evaluate_binary_classification
import pandas as pd
import numpy as np

def align_matrix(B_est, p):
    """
    Un-shuffles the estimated matrix back to canonical order.
    """
    idx = np.argsort(p)
    return B_est[idx, :][:, idx]

def run_simulation(n_iterations=50, n1=1000, n2=50, n_nodes=10):
    """
    Refactored simulation runner. 
    Fixes the multiclass error by binarizing the Naive delta for E1 \ E2.
    """
    all_results = []
    
    for seed in range(n_iterations):
        rng = np.random.default_rng(seed)
        simulator = EdgePerturbationSimulator(n_nodes=n_nodes, rng=rng)
        
        # 1. Generate Truth (Canonical) and Shuffled Data (Blind)
        B1_true, B2_true, delta_true = simulator.create_graphs(edge_prob=0.3, n_positives=2)
        df1, p1 = simulator.simulate_data(B1_true, n1)
        df2, p2 = simulator.simulate_data(B2_true, n2)
        
        # 2. Discovery on Shuffled Data
        comparator = CausalComparator(df1, df2)
        
        # Define specific execution and transformation logic for each method
        methods_config = ["Naive", "Bootstrap", "RSBS"]
        
        for name in methods_config:
            # 3. Execute method and retrieve raw results
            if name == "Naive":
                comparator.estimate_naive()
                optimize = False
            elif name == "Bootstrap":
                comparator.estimate_bootstrap(n_sampling=10)
                optimize = True
            else: # RSBS
                comparator.estimate_rsbs(n_sampling=10, seed=seed)
                optimize = True
            
            # 4. Alignment: Map System 1 and System 2 back to canonical space
            B1_aligned = align_matrix(comparator.freq_i, p1)
            B2_aligned = align_matrix(comparator.freq_j, p2)
            
            # 5. Transform to Difference Classification (E1 \ E2)
            if not optimize:
                # For Naive, we must be strictly binary {0, 1}
                # Edge exists in 1 AND NOT in 2
                delta_scores = ((B1_aligned == 1) & (B2_aligned == 0)).astype(int)
            else:
                # For Bootstrap/RSBS, the score is the difference in selection probs
                # Ranges from -1 to 1; optimize loop in metrics.py handles this correctly
                delta_scores = B1_aligned - B2_aligned
            
            # 6. Evaluate using metrics.py
            scores = evaluate_binary_classification(
                y_true=delta_true, 
                y_scores=delta_scores, 
                optimize=optimize
            )
            
            scores.update({"method": name, "seed": seed})
            all_results.append(scores)
            
        print(f"Iteration {seed+1}/{n_iterations} complete.")
            
    return pd.DataFrame(all_results)
df = run_simulation()

Iteration 1/50 complete.
Iteration 2/50 complete.
Iteration 3/50 complete.
Iteration 4/50 complete.
Iteration 5/50 complete.
Iteration 6/50 complete.
Iteration 7/50 complete.
Iteration 8/50 complete.
Iteration 9/50 complete.
Iteration 10/50 complete.
Iteration 11/50 complete.
Iteration 12/50 complete.
Iteration 13/50 complete.
Iteration 14/50 complete.
Iteration 15/50 complete.
Iteration 16/50 complete.
Iteration 17/50 complete.
Iteration 18/50 complete.
Iteration 19/50 complete.
Iteration 20/50 complete.
Iteration 21/50 complete.
Iteration 22/50 complete.
Iteration 23/50 complete.
Iteration 24/50 complete.
Iteration 25/50 complete.
Iteration 26/50 complete.
Iteration 27/50 complete.
Iteration 28/50 complete.
Iteration 29/50 complete.
Iteration 30/50 complete.
Iteration 31/50 complete.
Iteration 32/50 complete.
Iteration 33/50 complete.
Iteration 34/50 complete.
Iteration 35/50 complete.
Iteration 36/50 complete.
Iteration 37/50 complete.
Iteration 38/50 complete.
Iteration 39/50 compl

In [128]:
df.head(10)

,auc_roc,aupr,best_f1,best_threshold,precision,recall,method,seed
0,1.000000,1.000000,1.000000,N/A,1.000000,1.0,Naive,0
1,1.000000,1.000000,1.000000,0.505051,1.000000,1.0,Bootstrap,0
2,1.000000,1.000000,1.000000,0.505051,1.000000,1.0,RSBS,0
3,0.989796,0.500000,0.666667,N/A,0.500000,1.0,Naive,1
4,0.997449,0.833333,0.800000,0.606061,0.666667,1.0,Bootstrap,1
5,0.989796,0.700000,0.666667,0.40404,1.000000,0.5,RSBS,1
6,0.974490,0.285714,0.444444,N/A,0.285714,1.0,Naive,2
7,1.000000,1.000000,1.000000,0.707071,1.000000,1.0,Bootstrap,2
8,0.992347,0.583333,0.666667,0.40404,0.500000,1.0,RSBS,2
9,0.714286,0.072500,0.200000,N/A,0.125000,0.5,Naive,3


In [129]:
df.to_csv("res_test.csv", index=None)